# ETF AI ハンズオン
## Part 2: Cortex Analyst（自然言語 to SQL）

このノートブックでは、**Semantic View** の作成と **Cortex Analyst** による
自然言語クエリを体験します。

### このパートで体験できること

1. **Semantic View の作成**: ポートフォリオデータの「意味」を定義する
2. **Cortex Analyst のテスト**: 日本語でETFポートフォリオを分析する
3. **Snowsight UI でのデモ**: ノーコードでポートフォリオ分析を体験する

### 🚀 体験ポイント

> **「SQLを書かずに、自然言語でポートフォリオ分析！」**
>
> 「含み損が大きいポートフォリオはどれ？」「インカム系ETFの比重が高いポートフォリオは？」
> と入力するだけで、複数テーブルを結合した複雑なSQLが自動生成・実行されます。

### 前提条件
- `setup.sql` と `part1_data_setup.ipynb` の実行が完了していること

> ⏱️ **このパートの目安時間: 35分**

In [ ]:
USE DATABASE ETF_AI_HANDSON_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. データ構造の確認

Semantic View を作成する前に、対象テーブルの構造と関係を把握します。

> ⏱️ **目安: 5分**

In [ ]:
-- テーブル間の関係を確認（ER図的な俯瞰）
-- DIM_PORTFOLIO ─→ FACT_HOLDINGS ←─ DIM_ETF
--                        ↓
--                 FACT_PERFORMANCE

SELECT 'DIM_PORTFOLIO' AS "テーブル", '5件' AS "レコード数", 'ポートフォリオマスタ' AS "説明"
UNION ALL SELECT 'DIM_ETF',          '10件', 'ETF銘柄マスタ'
UNION ALL SELECT 'FACT_HOLDINGS',    '21件', '保有ETF明細（最新）'
UNION ALL SELECT 'FACT_PERFORMANCE', '48件', '月次パフォーマンス実績';

In [ ]:
-- 各テーブルのカラム一覧を確認（Semantic View 設計のために）
SELECT
    TABLE_NAME  AS "テーブル",
    COLUMN_NAME AS "カラム名",
    DATA_TYPE   AS "型",
    COMMENT     AS "説明"
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'
  AND TABLE_NAME IN ('DIM_ETF', 'DIM_PORTFOLIO', 'FACT_HOLDINGS', 'FACT_PERFORMANCE')
ORDER BY TABLE_NAME, ORDINAL_POSITION;

## 2. Semantic View とは

**Semantic View** は、Cortex Analyst が「テーブルの意味」を理解するための
**ビジネスメタデータ定義** です。通常のデータベースには「カラム名」しかありませんが、
Semantic View で「このカラムは何を意味するか」をAIに教えます。

### Semantic View の構成要素

| 要素 | 役割 | 例 |
|---|---|---|
| `TABLES` | 対象テーブルとその説明・主キー | `PORTFOLIO AS DIM_PORTFOLIO` |
| `RELATIONSHIPS` | テーブル間の関係（外部キー） | `PORTFOLIO → HOLDINGS (PORTFOLIO_ID)` |
| `SYNONYMS` | テーブル・カラムの別名（日本語対応） | `PORTFOLIOS` の同義語に `ポートフォリオ` |
| `FACTS` | 数値データの説明 | `WEIGHT_PCT` は `ETF組入比重（%）` |
| `DIMENSIONS` | カテゴリデータの説明 | `CATEGORY` は `ETFカテゴリ` |
| `METRICS` | 集計指標の定義 | `SUM(MARKET_VALUE_JPY)` は `保有ETF時価合計` |

> ⏱️ **目安: 15分**

### 2-1. ポートフォリオ分析 Semantic View の作成

以下の4つのテーブルを統合した Semantic View を作成します：

- `DIM_PORTFOLIO`: ポートフォリオマスタ（主軸）
- `DIM_ETF`: ETF銘柄マスタ
- `FACT_HOLDINGS`: 保有ETF明細
- `FACT_PERFORMANCE`: 月次パフォーマンス実績

In [ ]:
-- =============================================
-- Semantic View: ETFポートフォリオ分析
-- =============================================
CREATE OR REPLACE SEMANTIC VIEW ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW
TABLES (
    PORTFOLIOS AS ETF_AI_HANDSON_DB.DEMO_SCHEMA.DIM_PORTFOLIO
        PRIMARY KEY (PORTFOLIO_ID)
        WITH SYNONYMS = ('ポートフォリオ', '運用ファンド', 'ファンド', 'PF')
        COMMENT = 'ポートフォリオマスタ。5つの運用ポートフォリオの属性・目標・AUM情報',

    ETF_MASTER AS ETF_AI_HANDSON_DB.DEMO_SCHEMA.DIM_ETF
        PRIMARY KEY (ETF_CODE)
        WITH SYNONYMS = ('ETF', 'ファンド銘柄', '銘柄', 'ETF銘柄')
        COMMENT = 'ETF銘柄マスタ。10本のETF情報',

    HOLDINGS AS ETF_AI_HANDSON_DB.DEMO_SCHEMA.FACT_HOLDINGS
        PRIMARY KEY (HOLDING_ID)
        WITH SYNONYMS = ('保有明細', '保有ETF', '組入ETF', 'ポジション')
        COMMENT = '保有ETF明細。各ポートフォリオの組入ETFと時価・含み損益',

    PERFORMANCE AS ETF_AI_HANDSON_DB.DEMO_SCHEMA.FACT_PERFORMANCE
        PRIMARY KEY (PERFORMANCE_ID)
        WITH SYNONYMS = ('パフォーマンス', '運用実績', 'リターン')
        COMMENT = '月次パフォーマンス実績。各ポートフォリオの月次リターン・シャープレシオ・ドローダウン'
)
RELATIONSHIPS (
    PORTFOLIOS_TO_HOLDINGS AS
        PORTFOLIOS(PORTFOLIO_ID) REFERENCES HOLDINGS(PORTFOLIO_ID),
    HOLDINGS_TO_ETF AS
        HOLDINGS(ETF_CODE) REFERENCES ETF_MASTER(ETF_CODE),
    PORTFOLIOS_TO_PERFORMANCE AS
        PORTFOLIOS(PORTFOLIO_ID) REFERENCES PERFORMANCE(PORTFOLIO_ID)
)
FACTS (
    HOLDINGS.WEIGHT_PCT AS WEIGHT_PCT
        WITH SYNONYMS = ('組入比重', '比重', 'ウェイト', 'ETF比重')
        COMMENT = 'ポートフォリオ内のETF組入比重（%）。高いほど影響が大きい',

    HOLDINGS.MARKET_VALUE_JPY AS MARKET_VALUE_JPY
        WITH SYNONYMS = ('時価評価額', '時価', '評価額', '保有金額')
        COMMENT = '保有ETFの時価評価額（円）',

    HOLDINGS.UNREALIZED_GAIN_JPY AS UNREALIZED_GAIN_JPY
        WITH SYNONYMS = ('含み損益', '含み益', '含み損', '評価損益')
        COMMENT = '含み損益額（円）。プラスが含み益、マイナスが含み損',

    HOLDINGS.UNREALIZED_GAIN_PCT AS UNREALIZED_GAIN_PCT
        WITH SYNONYMS = ('含み損益率', '含み益率', '含み損率', '損益率')
        COMMENT = '含み損益率（%）',

    PERFORMANCE.RETURN_1M_PCT AS RETURN_1M_PCT
        WITH SYNONYMS = ('1ヶ月リターン', '月次リターン', '先月比', '月間騰落率')
        COMMENT = '1ヶ月リターン（%）',

    PERFORMANCE.RETURN_1Y_PCT AS RETURN_1Y_PCT
        WITH SYNONYMS = ('1年リターン', '年率リターン', '年間リターン', '1年騰落率')
        COMMENT = '1年間のトータルリターン（%）',

    PERFORMANCE.SHARPE_RATIO AS SHARPE_RATIO
        WITH SYNONYMS = ('シャープレシオ', 'シャープ比率', 'リスク調整後リターン')
        COMMENT = 'シャープレシオ。リスク1単位あたりの超過リターン。高いほど効率的な運用',

    PERFORMANCE.VOLATILITY_1Y AS VOLATILITY_1Y
        WITH SYNONYMS = ('ボラティリティ', '価格変動率', 'リスク', '標準偏差')
        COMMENT = '1年間の価格変動率（年率%）。高いほどリスクが大きい',

    PERFORMANCE.MAX_DRAWDOWN AS MAX_DRAWDOWN
        WITH SYNONYMS = ('最大ドローダウン', '最大下落率', '最大損失')
        COMMENT = '過去の最大下落率（%）。マイナス値で表示',

    ETF_MASTER.AUM_BILLION_JPY AS AUM_BILLION_JPY
        WITH SYNONYMS = ('純資産残高', 'ETF規模', '純資産', 'AUM')
        COMMENT = 'ETFの純資産残高（億円）',

    ETF_MASTER.EXPENSE_RATIO AS EXPENSE_RATIO
        WITH SYNONYMS = ('コスト', '信託報酬', '実質コスト', '経費率')
        COMMENT = 'ETFの実質コスト（年率）',

    PORTFOLIOS.AUM_MILLION_JPY AS PORTFOLIO_AUM
        WITH SYNONYMS = ('ポートフォリオ規模', '運用資産額', 'ポートフォリオAUM')
        COMMENT = 'ポートフォリオの運用資産額（百万円）',

    PORTFOLIOS.TARGET_RETURN AS TARGET_RETURN
        WITH SYNONYMS = ('目標リターン', '期待リターン', '運用目標')
        COMMENT = '年間目標リターン（%）'
)
DIMENSIONS (
    PORTFOLIOS.PORTFOLIO_NAME AS PORTFOLIO_NAME
        WITH SYNONYMS = ('ポートフォリオ名', 'PF名', 'ファンド名')
        COMMENT = 'ポートフォリオの名称',

    PORTFOLIOS.MANAGER_NAME AS MANAGER_NAME
        WITH SYNONYMS = ('PM', 'ポートフォリオマネージャー', '運用担当者', '担当PM')
        COMMENT = '担当ポートフォリオマネージャー名',

    PORTFOLIOS.RISK_PROFILE AS RISK_PROFILE
        WITH SYNONYMS = ('リスクプロファイル', 'リスク区分', '運用スタイル', 'リスク方針')
        COMMENT = 'リスクプロファイル（積極/バランス/安定）',

    ETF_MASTER.ETF_NAME_SHORT AS ETF_NAME
        WITH SYNONYMS = ('ETF名称', '銘柄名', 'ファンド名称')
        COMMENT = 'ETFの略称',

    ETF_MASTER.CATEGORY AS ETF_CATEGORY
        WITH SYNONYMS = ('ETFカテゴリ', 'カテゴリ', '種別', 'ETF種類')
        COMMENT = 'ETFのカテゴリ（コア/成長テーマ/インカム/コモディティ）',

    ETF_MASTER.ASSET_CLASS AS ASSET_CLASS
        WITH SYNONYMS = ('資産クラス', '投資対象', 'アセットクラス')
        COMMENT = '投資資産クラス（日本株式/米国株式/J-REIT/債券）',

    ETF_MASTER.THEME AS THEME
        WITH SYNONYMS = ('テーマ', '投資テーマ', '運用テーマ')
        COMMENT = '投資テーマ（半導体/高配当/グローバルリーダー等）',

    PERFORMANCE.PERF_DATE AS PERF_DATE
        WITH SYNONYMS = ('評価日', '基準日', '月末日')
        COMMENT = 'パフォーマンス評価月末日'
)
METRICS (
    PORTFOLIOS.COUNT_PORTFOLIOS AS COUNT(PORTFOLIOS.PORTFOLIO_ID)
        WITH SYNONYMS = ('ポートフォリオ数', '運用ファンド数')
        COMMENT = 'ポートフォリオ総数',

    PORTFOLIOS.TOTAL_AUM AS SUM(PORTFOLIOS.AUM_MILLION_JPY)
        WITH SYNONYMS = ('総運用資産', '合計AUM', 'AUM合計')
        COMMENT = '全ポートフォリオの合計運用資産額（百万円）',

    HOLDINGS.TOTAL_MARKET_VALUE AS SUM(HOLDINGS.MARKET_VALUE_JPY)
        WITH SYNONYMS = ('保有時価合計', '総時価評価額', '時価総額')
        COMMENT = '全保有ETFの時価評価合計（円）',

    HOLDINGS.TOTAL_UNREALIZED_GAIN AS SUM(HOLDINGS.UNREALIZED_GAIN_JPY)
        WITH SYNONYMS = ('含み損益合計', '評価損益合計', '総含み損益')
        COMMENT = '全保有ETFの含み損益合計（円）',

    HOLDINGS.AVG_WEIGHT AS AVG(HOLDINGS.WEIGHT_PCT)
        WITH SYNONYMS = ('平均比重', '平均組入比率')
        COMMENT = 'ETFの平均組入比重（%）'
);

ALTER SEMANTIC VIEW ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW
    SET COMMENT = 'ETFポートフォリオ分析用 Semantic View。保有ETF・パフォーマンス・ETFマスタを統合し、ポートフォリオマネージャーが自然言語で分析できる環境を提供';

SELECT 'PORTFOLIO_ANALYTICS_VIEW created!' AS STATUS;

In [ ]:
-- 作成された Semantic View の確認
SHOW SEMANTIC VIEWS IN SCHEMA ETF_AI_HANDSON_DB.AI;

## 3. Cortex Analyst のテスト（REST API経由）

Snowflake Notebook から Cortex Analyst REST API を直接呼び出してテストします。

> ⏱️ **目安: 15分**

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

SEMANTIC_VIEW = "ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW"

def ask_cortex_analyst(question: str) -> dict:
    """Cortex Analyst REST API を呼び出す"""
    request_body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_view": SEMANTIC_VIEW
    }
    resp = session.connection.rest.request(
        url="/api/v2/cortex/analyst/message",
        body=request_body,
        method="post",
        client="rest",
        timeout=60,
    )
    return resp

def ask_and_run(question: str):
    """Cortex Analyst に質問し、生成されたSQLを実行して結果を表示"""
    print(f"質問: {question}")
    print("=" * 70)
    result = ask_cortex_analyst(question)
    generated_sql = None
    for msg in result.get("message", {}).get("content", []):
        if msg["type"] == "text":
            print(msg["text"])
        elif msg["type"] == "sql":
            generated_sql = msg["statement"]
            print(f"\n【自動生成されたSQL】\n{generated_sql}")
    if generated_sql:
        print("\n【クエリ結果】")
        df = session.sql(generated_sql).to_pandas()
        display(df)
    return result

print("Cortex Analyst クライアント準備完了")

In [ ]:
# Q1: ポートフォリオ別の含み損益ランキング
ask_and_run("各ポートフォリオの含み損益合計を多い順にランキングしてください")

In [ ]:
# Q2: インカム系ETFの比重
ask_and_run("インカム系ETFの組入比重が最も高いポートフォリオはどれですか？")

In [ ]:
# Q3: シャープレシオが最も高いポートフォリオ
ask_and_run("2025年12月末時点でシャープレシオが最も高いポートフォリオを教えてください")

In [ ]:
# Q4: グローバルリーダーズETFの保有状況
ask_and_run("グローバルリーダーズETF（2641）を保有しているポートフォリオの一覧と、それぞれの比重を教えてください")

In [ ]:
# Q5: 目標リターンを上回ったポートフォリオ（自由に試してみてください）
ask_and_run("2025年の年間リターンが目標リターンを上回ったポートフォリオはどれですか？")

## 4. Snowsight の Cortex Analyst UI でのテスト

### GUI アクセス手順

1. **Snowsight** にログイン
2. 左メニュー「**AI & ML**」→「**Cortex Analyst**」をクリック
3. 「**+ New**」ボタンをクリック
4. Semantic View として `ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW` を選択
5. 下部のチャットボックスに質問を入力

### テスト質問例

以下の質問を Cortex Analyst UI に入力して結果を確認してください。

---

**Q1**: 「各ポートフォリオの含み損益合計を多い順にリストアップしてください」

**Q2**: 「インカム系ETFの比重が最も高いポートフォリオはどれですか？」

**Q3**: 「2025年のシャープレシオが最も高いポートフォリオとその値を教えてください」

**Q4**: 「グローバルリーダーズETF（2641）を保有しているポートフォリオを全て教えて」

**Q5**: 「目標リターンを超えたポートフォリオの一覧と超過幅を教えてください」

**Q6**: 「テクノロジー成長ポートフォリオの2026年1〜3月の月次リターン推移を見せて」

**Q7**: 「純資産残高が最も大きいETF上位5本と、それを保有するポートフォリオを教えて」

---

> 💡 **ポイント**: Semantic View に `SYNONYMS` を設定しているため、
> 「ポートフォリオ」「運用ファンド」「PF」などどれで質問しても同じ結果が返ります！

In [ ]:
-- 【手動確認 Q1】各ポートフォリオの含み損益合計
SELECT
    dp.PORTFOLIO_NAME AS "ポートフォリオ名",
    dp.MANAGER_NAME   AS "PM",
    TO_CHAR(SUM(h.UNREALIZED_GAIN_JPY) / 100000000, '9,999.0') || '億円' AS "含み損益合計",
    TO_CHAR(SUM(h.MARKET_VALUE_JPY) / 100000000, '99,999.0') || '億円'   AS "時価評価額合計"
FROM FACT_HOLDINGS h
JOIN DIM_PORTFOLIO dp ON h.PORTFOLIO_ID = dp.PORTFOLIO_ID
GROUP BY dp.PORTFOLIO_NAME, dp.MANAGER_NAME
ORDER BY SUM(h.UNREALIZED_GAIN_JPY) DESC;

In [ ]:
-- 【手動確認 Q3】2025年通年シャープレシオランキング
SELECT
    dp.PORTFOLIO_NAME  AS "ポートフォリオ名",
    dp.RISK_PROFILE    AS "リスクプロファイル",
    p.RETURN_1Y_PCT    AS "1年リターン(%)",
    p.VOLATILITY_1Y    AS "ボラティリティ(%)",
    p.SHARPE_RATIO     AS "シャープレシオ",
    p.MAX_DRAWDOWN     AS "最大DD(%)"
FROM FACT_PERFORMANCE p
JOIN DIM_PORTFOLIO dp ON p.PORTFOLIO_ID = dp.PORTFOLIO_ID
WHERE p.PERF_DATE = '2025-12-31'
  AND p.ETF_CODE IS NULL
ORDER BY p.SHARPE_RATIO DESC;

In [ ]:
-- 【手動確認 Q5】目標リターンを超えたポートフォリオ
SELECT
    dp.PORTFOLIO_NAME      AS "ポートフォリオ名",
    dp.TARGET_RETURN       AS "目標リターン(%)",
    p.RETURN_1Y_PCT        AS "実績リターン(%)",
    p.RETURN_1Y_PCT - dp.TARGET_RETURN AS "超過幅(%)",
    CASE WHEN p.RETURN_1Y_PCT >= dp.TARGET_RETURN THEN '達成' ELSE '未達' END AS "達成状況"
FROM FACT_PERFORMANCE p
JOIN DIM_PORTFOLIO dp ON p.PORTFOLIO_ID = dp.PORTFOLIO_ID
WHERE p.PERF_DATE = '2025-12-31'
  AND p.ETF_CODE IS NULL
ORDER BY (p.RETURN_1Y_PCT - dp.TARGET_RETURN) DESC;

## まとめ

Part 2 完了！Cortex Analyst のセットアップが完了しました。

### 作成したオブジェクト

| オブジェクト | 種別 | 場所 |
|---|---|---|
| `PORTFOLIO_ANALYTICS_VIEW` | Semantic View | `ETF_AI_HANDSON_DB.AI` |

### Semantic View の威力

- **同義語対応**: 「ポートフォリオ」「PF」「運用ファンド」どれで質問しても同じ結果
- **複数テーブル結合**: 「グローバルリーダーズを持つPFのシャープレシオは？」も自動的にJOIN
- **ビジネス用語変換**: `WEIGHT_PCT` を「組入比重」「ウェイト」として認識
- **日本語対応**: 日本語でも高精度に自然言語 to SQL が動作

### よくある質問

**Q: Semantic View と通常のVIEWの違いは？**
A: 通常のVIEWはSQLで定義されたデータ変換ですが、Semantic ViewはAIへのメタデータ定義です。
   SQLは含まず、テーブルの「意味」のみを定義します。

**次のステップ**: `part3_cortex_search.ipynb` で
ETFドキュメントの Cortex Search Service を構築してください。